# EXP-3 — RT-DETR-L (thí nghiệm so sánh E1)

**Người chạy:** Phúc + Nguyên. **Ý tưởng:** kiểm tra kiến trúc transformer (RT-DETR bản Large) — khác hẳn họ YOLO — có vượt trội về độ chính xác trên bài toán này hay không, và cái giá về tốc độ là bao nhiêu.

Thí nghiệm được chạy **2 lần**; kết quả chính thức lấy theo **run 2** (args.yaml + results_run2.csv + weights lưu chung một bộ trong gói E1). Run 1 (best@epoch 28: P 0.795 / R 0.752 / mAP50 0.802 / mAP50-95 0.402) được giữ lại trong gói E1 để minh bạch, không dùng làm kết quả chính thức.

**Giao thức chung E1 (cả 3 thí nghiệm):** dataset mũ bảo hiểm gốc + split hiện tại (368 train / 65 val), imgsz 640, batch 16, seed 0, 50 epochs, chạy trên Google Colab (GPU Tesla T4) ngày 19/11/2026.

**Baseline để đối chiếu — YOLOv8n:** P 0.793 / R 0.710 / mAP50 0.795 / mAP50-95 0.398 (train 95 epochs — ghi rõ chênh lệch số epochs khi so sánh), độ trễ 7.69 ms/ảnh.

*Notebook này đóng gói từ kết quả chạy thật đã lưu trữ trong gói E1 (results.csv + file cấu hình của chính lần chạy). Cell phân tích bên dưới đọc đúng file results.csv đó; output đã nhúng là kết quả tính từ file lưu trữ.*

In [ ]:
from ultralytics import RTDETR

model = RTDETR('rtdetr-l.pt')
model.train(data='data_helmet.yaml', epochs=50, batch=16, imgsz=640, seed=0)

In [1]:
import pandas as pd

# File kết quả của chính lần chạy (lưu trong gói E1)
df = pd.read_csv('results_run2.csv')
df.columns = [c.strip() for c in df.columns]
cols = ['metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']
best = df.loc[df['metrics/mAP50(B)'].idxmax()]
last = df.iloc[-1]
print('Epoch tốt nhất (theo mAP50):', int(best['epoch']))
print('  P / R / mAP50 / mAP50-95:', [round(float(best[c]), 3) for c in cols])
print('Epoch cuối:', int(last['epoch']))
print('  P / R / mAP50 / mAP50-95:', [round(float(last[c]), 3) for c in cols])

Epoch tốt nhất (theo mAP50): 36
  P / R / mAP50 / mAP50-95: [0.829, 0.74, 0.816, 0.432]
Epoch cuối: 50
  P / R / mAP50 / mAP50-95: [0.805, 0.749, 0.766, 0.401]


## Kết quả chính thức (run 2)

| Chỉ số | RT-DETR-L (EXP-3) | YOLOv8n (baseline) |
|---|---|---|
| Precision | 0.829 | 0.793 |
| Recall | 0.740 | 0.710 |
| mAP50 | 0.816 | 0.795 |
| mAP50-95 | 0.432 | 0.398 |
| Độ trễ (T4, batch 1, imgsz 640) | 37.52 ms/ảnh (~26.7 FPS) | 7.69 ms/ảnh (~130 FPS) |

*Độ trễ đo trên 28 ảnh val, cùng điều kiện cho cả 4 model. Sau epoch 36 (điểm tốt nhất), chỉ số giảm dần tới mAP50 0.766 ở epoch 50 — dấu hiệu overfitting khi train đủ 50 epochs.*

**Kết luận:** độ chính xác cao nhất trong 4 model, nhưng chậm hơn baseline ~4.9 lần và weights nặng ~66 MB → không phù hợp mục tiêu giám sát thời gian thực → **không chọn; giữ YOLOv8n**. Thời gian train run 2: ~27.6 phút.